In [1]:
!pip uninstall -qqy kfp jupyterlab libpysal thinc spacy fastai ydata-profiling google-cloud-bigquery google-generativeai
# Install langgraph and the packages used in this lab.
!pip install -qU 'langgraph==0.3.21' 'langgraph-prebuilt==0.1.7'
!pip uninstall -qqy jupyterlab kfp  # Remove unused conflicting packages
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.9/100.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 13.1 MB/s et

# Note for judges


**Note for Judges:This notebook uses a pre-trained ResNet50 model to avoid long training times during evaluation.The model was trained on the ISIC dataset, and weights are loaded from a saved file.The focus is on generative AI capabilities: structured output, RAG, and a LangGraph chatbot.**

**HOW TO USE: the test-images-dataset contains images(img_1.jpg, img_2.jpg.... and so on), while interacting with the bot you need to input the name of the image(eg. img_32, img_1) and optionally you can put the anatom_site as well(eg. neck, torso) in the start_chat() function, after this the chatbot will predict the skin lesion conditon and give you the diagnosis**

In [2]:
import os
import pandas as pd
import numpy as np
# Suppress TensorFlow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from langchain.prompts import PromptTemplate
from pydantic import BaseModel
from typing import List, Dict
from langgraph.graph import StateGraph, END
import json
from pprint import pprint

E0000 00:00:1745188574.363958      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745188574.441124      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Define dataset directory - replace 'your_dataset_name' with your actual dataset slug
DATASET_DIR = "/kaggle/input/isic-skin-lesion-dataset"
benign_dir = os.path.join(DATASET_DIR, "Benign")
malignant_dir = os.path.join(DATASET_DIR, "Malignant")
benign_metadata_path = os.path.join(benign_dir, "metadata.csv")
malignant_metadata_path = os.path.join(malignant_dir, "metadata.csv")
IMAGES_DIR = "/kaggle/input/test-images-dataset/test_images/"

In [4]:
df1 = pd.read_csv(benign_metadata_path)
df2 = pd.read_csv(malignant_metadata_path)

df1 = df1[['isic_id', 'anatom_site_general', 'benign_malignant', 'diagnosis_1', 'diagnosis']]
df2 = df2[['isic_id', 'anatom_site_general', 'benign_malignant', 'diagnosis_1', 'diagnosis']]

metadata = pd.concat([df1, df2], ignore_index=True)
print(metadata.shape)  # rows = rows1 + rows2

(21272, 5)


In [5]:
# Model1

# Check class distribution in 'diagnosis_4'
class_counts = metadata['diagnosis'].value_counts()
# print("Class distribution before merging:\n", class_counts)

# Merge classes with fewer than 2 samples into 'Other'
metadata['diagnosis_modified'] = metadata['diagnosis'].where(
    metadata['diagnosis'].isin(class_counts[class_counts >= 2].index), 'Other'
)



# Verify new class distribution
new_class_counts = metadata['diagnosis_modified'].value_counts()
# print("Class distribution after merging:\n", new_class_counts)

# Encode the modified target variable
le = LabelEncoder()
le.fit(metadata['diagnosis_modified'])
metadata['label'] = le.transform(metadata['diagnosis_modified'])

def get_image_path(row):
    if row['diagnosis_1'] == 'Benign':
        return os.path.join(benign_dir, row['isic_id'] + '.jpg')
    else:
        return os.path.join(malignant_dir, row['isic_id'] + '.jpg')

# Add image paths to the DataFrame
metadata['image_path'] = metadata.apply(get_image_path, axis=1)


# Filter out rows with invalid image paths
metadata = metadata[metadata['image_path'].notnull()]

# Verify image paths exist
missing_files = metadata[~metadata['image_path'].apply(os.path.exists)]
if not missing_files.empty:
    # print("Warning: Missing image files for the following entries:\n", missing_files[['isic_id', 'image_path']])
    # Remove rows with missing files
    metadata = metadata[metadata['image_path'].apply(os.path.exists)]
# print(f"Total samples after filtering: {len(metadata)}")


# Split data into training and validation sets with stratification
train_df, val_df = train_test_split(metadata, test_size=0.2, stratify=metadata['label'], random_state=42)


# Function to load and preprocess images with error handling
def load_and_preprocess_image(path, label):
    try:
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, [224, 224])
        image = image / 255.0  # Normalize pixel values to [0,1]
        return image, label
    except:
        # Return a placeholder to avoid crashing; these will be filtered out
        return tf.zeros((224, 224, 3)), label


train_dataset = tf.data.Dataset.from_tensor_slices((train_df['image_path'].values, train_df['label'].values))
train_dataset = train_dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_df['image_path'].values, val_df['label'].values))
val_dataset = val_dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dense(len(le.classes_), activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])



94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


**For the purpose of the competion, I have used a model with pretrained weights, since training would have taken lot of time, This CNN model is used to prdict the skin lesion condion which is then provided to the LLM model with optional information such as lesion_location to provide necessary next steps and treatment plan**

In [6]:
## we won't be running this cell as this was only used during the training of CNN, for now we 
## will use the pre-trained weights


# history = model.fit(train_dataset, epochs=4, validation_data=val_dataset)
# model.save('/kaggle/working/skin_lesion_model.h5')

In [7]:
model = tf.keras.models.load_model('/kaggle/input/pre-trained-weights/skin_lesion_model (1).h5')
print("Pre-trained model loaded successfully.")

Pre-trained model loaded successfully.


# **2nd part**

In [8]:
# Generative AI: RAG with ChromaDB

from google import genai
from google.genai import types

from IPython.display import Markdown

genai.__version__

'1.7.0'

In [9]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

In [10]:
DOCUMENT1 = "Surgical therapy of basal cell carcinomas (BCC) is based on complete excision of the neoplasm and its immediate suitable reconstruction. The aim of this work was to evaluate the possibility of creating a reconstructive algorithm in cases of scalp BCC, depending on the amplitude of the tumor. Basal cell carcinoma (BCC) is the most common type of cancer with an estimated 3.6 million cases diagnosed annually in the US alone. While most cases are treatable with low recurrence rates, 1-10\% progress to an advanced stage which can behave aggressively, leading to local destruction and posing substantial challenges in management. The pathogenesis often involves dysregulation of the patched/hedgehog protein family, a pivotal pathway targeted by recently approved therapies. Furthermore, the role of immunotherapy is evolving in this type of tumor as we learn more about tumor microenvironment dynamics. In recent years, there have been advancements in the therapeutic landscape of advanced BCC, offering patients new hope and options for managing this complex and potentially life-threatening condition. In this review, we aim to provide a comprehensive overview of this disease, including the risk factors, underlying pathogenesis, current treatment options of advanced disease, and the ongoing exploration and development of novel therapies. Basal cell carcinoma (BCC) is the most common cutaneous malignancy. Ultraviolet light is an important risk factor for the pathogenesis of BCCs; the vast majority are found in sun-exposed areas. BCCs occurring in the perianal or genital regions are seldom seen. Less than 1\% of all BCCs occur at these sites. Etiologic factors other than solar exposure must be taken into account for such cases. We report a rare case of BCC that was initially detected during a routine colonoscopy. This evidence-based summary covers basal cell carcinoma (BCC) and squamous cell carcinoma (SCC) of the skin and the related noninvasive lesion actinic keratosis (viewed by some pathologists as a variant of in situ SCC). BCC and SCC are both of epithelial origin. Although BCC and SCC are by far the most frequent types of nonmelanoma skin cancers, approximately 82 types of skin malignancies, with a wide range of clinical behaviors, fall into the category of nonmelanoma skin cancer. BCC is at least three times more common than SCC in nonimmunocompromised patients. It usually occurs on sun-exposed areas of skin, with the nose being the most common site. Although there are many different clinical presentations for BCC, the most characteristic type is the asymptomatic nodular or nodular ulcerative lesion that is elevated from the surrounding skin, has a pearly quality, and contains telangiectatic vessels. BCCs are composed of nonkeratinizing cells derived from the basal cell layer of the epidermis. They are slow growing and rarely metastasize. BCC has a tendency to be locally destructive and can result in serious deforming damage if left untreated or if local recurrences cannot be completely excised. High-risk areas for tumor recurrence after initial treatment include the central face (e.g., periorbital region, eyelids, nasolabial fold, or nose-cheek angle), postauricular region, pinna, ear canal, forehead, and scalp. Morpheaform type is a specific BCC subtype. This subtype typically appears as a scar-like, firm plaque. Because of indistinct clinical tumor margins, morpheaform type is difficult to treat adequately with traditional treatments. BCCs often have a characteristic mutation in the PTCH1 tumor suppressor gene, although the mechanism of carcinogenesis is not clear. People with chronic sun damage, history of sunburns, arsenic exposure, chronic cutaneous inflammation (as seen in long-standing skin ulcers), and previous radiation therapy are predisposed to the development of SCC. SCCs tend to occur on sun-exposed portions of the skin, such as the ears, lower lip, and dorsa of the hands. SCCs that develop from actinic keratosis on sun-exposed skin are less likely to metastasize and have a better prognosis than those that develop de novo, or on non–sun-exposed skin. SCCs are composed of keratinizing cells. These tumors are more aggressive than BCCs and have a range of growth, invasive, and metastatic potential. Prognosis is associated with the degree of differentiation, and tumor grade is reported as part of the staging system.[10] A four-grade system (G1–G4) is most common, but two- and three-grade systems may also be used. Mutations in the PTCH1 tumor suppressor gene have been reported in SCCs removed from patients with a prior history of multiple BCCs. SCC in situ (also called Bowen disease) is a noninvasive lesion. Distinguishing SCC in situ pathologically from a benign inflammatory process may be difficult.[1] The risk of development into invasive SCC is low, reportedly in the range of 3\% to 4\%. BCC and SCC are usually diagnosed on the basis of routine histopathology obtained from a shave, punch, incisional, or excisional biopsy"
DOCUMENT2 = "Basal-cell carcinoma (BCC), also known as basal-cell cancer, basalioma or rodent ulcer, is the most common type of skin cancer. It often appears as a painless raised area of skin, which may be shiny with small blood vessels running over it. It may also present as a raised area with ulceration. Basal-cell cancer grows slowly and can damage the tissue around it, but it is unlikely to spread to distant areas or result in death. Risk factors include exposure to ultraviolet light, having lighter skin, radiation therapy, long-term exposure to arsenic and poor immune-system function. Exposure to UV light during childhood is particularly harmful. Tanning beds have become another common source of ultraviolet radiation. Diagnosis often depends on skin examination, confirmed by tissue biopsy. It remains unclear whether sunscreen affects the risk of basal-cell cancer.[11] Treatment is typically by surgical removal.[2] This can be by simple excision if the cancer is small; otherwise, Mohs surgery is generally recommended. Other options include electrodesiccation and curettage, cryosurgery, topical chemotherapy, photodynamic therapy, laser surgery or the use of imiquimod, a topical immune-activating medication.[12] In the rare cases in which distant spread has occurred, chemotherapy or targeted therapy may be used. Basal-cell cancer accounts for at least 32\% of all cancers globally.[9][13] Of skin cancers other than melanoma, about 80\% are basal-cell cancers. In the United States, about 35\% of white males and 25\% of white females are affected by BCC at some point in their lives. Basal-cell carcinoma is named after the basal cells that form the lowest layer of the epidermis. It is thought to develop from the folliculo–sebaceous–apocrine germinative cells called trichoblasts (of note, trichoblastic carcinoma is a term sometimes used to refer to a rare type of aggressive skin cancer that may resemble a benign trichoblastoma, and can also closely resemble basal cell carcinoma). Individuals with basal-cell carcinoma typically present with a shiny, pearly skin nodule. However, superficial basal-cell cancer can present as a red patch similar to eczema. Infiltrative or morpheaform basal-cell cancers can present as a skin thickening or scar tissue – making diagnosis difficult without using tactile sensation and a skin biopsy. It is often difficult to visually distinguish basal-cell cancer from acne scar, actinic elastosis, and recent cryodestruction inflammation. The majority of basal-cell carcinomas occur on sun-exposed areas of the body. Basal-cell carcinoma cells appear similar to epidermal basal cells and are usually well differentiated. Surgery to remove the basal-cell carcinoma affected area and the surrounding skin is thought to be the most effective treatment.[40] A disadvantage with standard surgical excision is a reported higher recurrence rate of basal-cell cancers of the face,[41] especially around the eyelids,[42] nose, and facial structures.[43] There is no clear approach, nor clear research comparing the effectiveness of Mohs micrographic surgery versus surgical excision for BCC of the eye. For many new (primary) and all recurrent forms of basal cell carcinoma after previous surgery, especially on the head, neck, hands, feet, genitalia, and anterior legs (shins), Mohs surgery should be considered. An essential aspect of Mohs surgery is that the Mohs surgeon performs the surgery and personally reviews the Mohs pathology slides.[47] Most standard excisions done in an office setting are sent to an outside laboratory for standard bread loafing methods of processing.[49] With this method, it is likely that less than 5\% of the surgical margin is examined, as each slice of tissue is only 6 micrometres thick, about 3 to 4 serial slices are obtained per section, and only about 3 to 4 sections are obtained per specimen. Cryosurgery is an old modality for the treatment of many skin cancers. When accurately utilized with a temperature probe and cryotherapy instruments, it can result in a very good cure rate. Disadvantages include lack of margin control, tissue necrosis, over or under-treatment of the tumor, and long recovery time. Overall, there are sufficient data to consider cryosurgery as a reasonable treatment for BCC. There are no good studies, however, comparing cryosurgery with other modalities, particularly with Mohs surgery, excision, or electrodesiccation and curettage so no conclusion can be made whether cryosurgery is as efficacious as other methods. Electrodesiccation and curettage (EDC, also known as curettage and cautery, simply curettage)[53] is accomplished by using a round knife, or curette, to scrape away the soft cancer. The skin is then burned with an electric current. This further softens the skin, allowing for the knife to cut more deeply with the next layer of curettage. The cycle is repeated, with a safety margin of curettage of normal skin around the visible tumor. This cycle is repeated 3 to 5 times, and the free skin margin treated is usually 4 to 6 mm. Cure rate is very much user-dependent and depends also on the size and type of tumor. Infiltrative or morpheaform BCCs can be difficult to eradicate with EDC. Generally, this method is used on cosmetically unimportant areas like the trunk (torso). Some physicians believe that it is acceptable to utilize EDC on the face of elderly patients over the age of 70. However, with increasing life expectancy, such an objective criterion cannot be supported. The cure rate can vary, depending on the aggressiveness of the EDC and the free margin treated. Some advocate curettage alone without electrodesiccation, and with the same cure rate Some superficial cancers respond to local therapy with 5-fluorouracil, a chemotherapy agent. One can expect a great deal of inflammation with this treatment.[55] Chemotherapy often follows Mohs surgery to eliminate the residual superficial basal-cell carcinoma after the invasive portion is removed. 5-fluorouracil has received FDA approval. Removing the residual superficial tumor with surgery alone can result in large and difficult-to-repair surgical defects. One often waits a month or more after surgery before starting the immunotherapy or chemotherapy to make sure the surgical wound has adequately healed. Some people[who?] advocate the use of curettage (see EDC below) first, followed by chemotherapy. These experimental procedures are not standard care. Vismodegib and sonidegib are drugs approved for specially treating BCC, but are expensive and cannot be used in pregnant women. Itraconazole, traditionally an anti-fungal medication, has also garnered recent attention for its potential use in the treatment of BCC, especially those that cannot be removed surgically. Possessing anti-Hedgehog pathway activity, there is clinical evidence that itraconazole has some efficacy either alone or when combined with vismodegib/sonidegib for primary and recurrent BCC. There is one case report of efficacy in metastatic BCC. This technique uses the body's immune system to kill cancer cells. Improvement of the immune system works its way out up to the cancerous cells and treat the skin cancer. Topical treatment with 5\% Imiquimod cream (IMQ), with five applications per week for six weeks, has a reported 70–90\% success rate at reducing, even removing, the BCC [basal-cell carcinoma]. Imiquimod has received FDA approval and topical IMQ is approved by the European Medicines Agency for the treatment of small superficial basal-cell carcinoma.[53] Off-label use of imiquimod on invasive basal-cell carcinoma has been reported. Imiquimod may be used before surgery to reduce the size of the carcinoma. Research suggests that treatment using Euphorbia peplus, a common garden weed, may be effective.[58] Australian biopharmaceutical company Peplin[59] is developing this as topical treatment for BCC. Radiation therapy can be delivered either as external beam radiotherapy or as brachytherapy (mostly internal radiotherapy). Although radiotherapy is generally used in older patients who are not candidates for surgery, it is also used in cases where surgical excision will be disfiguring or difficult to reconstruct (especially on the tip of the nose, and the nostril rims). Radiation treatment with external radiation often takes as few as 5 visits to as many as 25 visits. Usually, the more visits scheduled for therapy, the less complication or damage is done to the normal tissue supporting the tumor. Radiotherapy can also be useful if surgical excision has been done incompletely or if the pathology report following surgery suggests a high risk of recurrence, for example, if nerve involvement has been demonstrated. The cure rate can be as high as 95\% for small tumors, or as low as 80\% for large tumors. A variation of an external brachytherapy is the epidermal radioisotope therapy (e.g. with 188Re in the form of the Rhenium-SCT). It is used in accordance with the general indications for brachytherapy and especially complex localisations or structures (e.g. earlobe) as well as the genitals. Photodynamic therapy (PDT) is a new modality for the treatment of basal-cell carcinoma, which is administered by application of photosensitizers to the target area. When these molecules are activated by light, they become toxic, therefore destroying the target cells. Methyl aminolevulinate is approved by the EU as a photosensitizer since 2001. This therapy is also used in other skin cancer types.[61] A 2008 study reported that PDT was a good treatment option for primary superficial BCCs and reasonable for primary low-risk nodular BCCs, but was a \"relatively poor\" option for high-risk lesions. Prognosis is excellent if the appropriate method of treatment is used in early primary basal-cell cancers. Recurrent cancers are much harder to cure, with a higher recurrence rate with any method of treatment. Although basal-cell carcinoma rarely metastasizes, it grows locally with invasion and destruction of local tissues. The cancer can impinge on vital structures like nerves and result in loss of sensation or loss of function or rarely death. The vast majority of cases can be successfully treated before serious complications occur. The recurrence rate for the above treatment options ranges from 50 percent to 1 percent or less."
DOCUMENT3 = "Squamous-cell carcinoma (SCC), also known as epidermoid carcinoma, comprises a number of different types of cancer that begin in squamous cells.[1] These cells form on the surface of the skin, on the lining of hollow organs in the body, and on the lining of the respiratory and digestive tracts. The squamous-cell carcinomas of different body sites can show differences in their presented symptoms, natural history, prognosis, and response to treatment. Human papillomavirus infection has been associated with SCCs of the oropharynx, lung, fingers, and anogenital region. About 90\%[4] of cases of head and neck cancer are due to SCC. Cutaneous squamous-cell carcinoma is the second most common skin cancer, accounting for over 1 million cases in the United States each year. Primary squamous-cell carcinoma of the thyroid shows an aggressive biological phenotype resulting in poor prognosis for patients. Esophageal cancer may be due to either esophageal squamous-cell carcinoma (ESCC) or adenocarcinoma (EAC). SCCs tend to occur closer to the mouth, while adenocarcinomas occur closer to the stomach. Dysphagia (difficulty swallowing, solids worse than liquids) and painful swallowing are common initial symptoms. If the disease is localized, surgical removal of the affected esophagus may offer the possibility of a cure. If the disease has spread, chemotherapy and radiotherapy are commonly used. When associated with the lung, it is typically a centrally located large-cell cancer (non-small-cell lung cancer). It often has a paraneoplastic syndrome causing ectopic production of parathyroid hormone-related protein, resulting in hypercalcemia, but paraneoplastic syndrome is more commonly associated with small-cell lung cancer. It is primarily due to smoking. Human papillomavirus (HPV), primarily HPV 16 and 18, are strongly implicated in the development of SCC of the penis. Three carcinomas in situ are associated with SCCs of the penis. When associated with the prostate, squamous-cell carcinoma is very aggressive in nature.[10] It is difficult to detect as no increase in prostate-specific antigen levels is seen, meaning that the cancer is often diagnosed at an advanced stage. Squamous cell carcinoma of the vagina spreads slowly and usually stays near the vagina, but may spread to the lungs and liver. This is the most common type of vaginal cancer. Ovarian squamous cell carcinoma (oSCC) or squamous ovarian carcinoma (SOC) is a rare tumor that accounts for 1\% of ovarian cancers. Conjunctival squamous cell carcinoma and corneal intraepithelial neoplasia comprise ocular surface squamous neoplasia (OSSN). Studies have found evidences for an association between diet and skin cancers, including SCC. The consumption of high-fat dairy foods increases SCC tumor risk in people with previous skin cancer. Green leafy vegetables may help prevent development of subsequent SCC and multiple studies found that raw vegetables and fruits are significantly protective against SCC risk.[27][28] On the other hand, consumption of whole milk, yogurt, and cheese may increase SCC risk in susceptible people.[29] In addition, meat and fat dietary pattern can increase the risk of SCC in people without a history of SCC, but the association is again more prominent in people with a history of skin cancer.[30] Tobacco smoking and a dietary pattern characterized by high beer and liquor intake also increase the risk of SCC significantly. Chemotherapy is the treatment of cancer with drugs (\"anticancer drugs\") that can destroy cancer cells. Chemotherapy can be given in a variety of ways such as injections into the muscles, skin, artery, or vein, or it could even be taken by mouth in the form of a pill.[11] In current usage, the term \"chemotherapy\" usually refers to cytotoxic drugs which affect rapidly dividing cells in general, in contrast with targeted therapy (see below). Chemotherapy drugs interfere with cell division in various possible ways, e.g. with the duplication of DNA or the separation of newly formed chromosomes. Most forms of chemotherapy target all rapidly dividing cells and are not specific to cancer cells, although some degree of specificity may come from the inability of many cancer cells to repair DNA damage, while normal cells generally can. Hence, chemotherapy has the potential to harm healthy tissue, especially those tissues that have a high replacement rate (e.g. intestinal lining). These cells usually repair themselves after chemotherapy. Radiation therapy (radiotherapy) is the use of ionizing radiation to kill cancer cells and shrink tumors by damaging their DNA causing cellular death.[9] Radiation therapy can either damage DNA directly or create charged particles (free radicals) within the cells that can in turn damage the DNA. Radiation therapy can be administered externally via external beam radiotherapy or internally via brachytherapy. The effects of radiation therapy are localised and confined to the region being treated. Although radiation damages both cancer cells and normal cells, most normal cells can recover from the effects of radiation and function properly. The goal of radiation therapy is to damage as many cancer cells as possible, while limiting harm to nearby healthy tissue. Hence, it is given in many fractions, allowing healthy tissue to recover between fractions. Cancer immunotherapy refers to a diverse set of therapeutic strategies designed to induce the patient's own immune system to fight the tumor. Contemporary methods for generating an immune response against tumors include intravesical BCG immunotherapy for superficial bladder cancer, and use of interferons and other cytokines to induce an immune response in renal cell carcinoma and melanoma patients. Cancer vaccines to generate specific immune responses are the subject of intensive research for a number of tumors, notably malignant melanoma and renal cell carcinoma. Sipuleucel-T is a vaccine-like strategy for prostate cancer in which dendritic cells from the patient are loaded with prostatic acid phosphatase peptides to induce a specific immune response against prostate-derived cells. It gained FDA approval in 2010. The growth of some cancers can be inhibited by providing or blocking certain hormones. Common examples of hormone-sensitive tumors include certain types of breast and prostate cancers. Blocking estrogen or testosterone is often an important additional treatment. In certain cancers, administration of hormone agonists, such as progestogens may be therapeutically beneficial. Although the side effects from hormone therapy vary depending on the type, patients can experience symptoms such as hot flashes, nausea, and fatigue. Synthetic lethality arises when a combination of deficiencies in the expression of two or more genes leads to cell death, whereas a deficiency in only one of these genes does not. The deficiencies can arise through mutations, epigenetic alterations or inhibitors of one or both of the genes. Cancer cells are frequently deficient in a DNA repair gene. This DNA repair defect either may be due to mutation or, often, epigenetic silencing (see epigenetic silencing of DNA repair). If this DNA repair defect is in one of seven DNA repair pathways (see DNA repair pathways), and a compensating DNA repair pathway is inhibited, then the tumor cells may be killed by synthetic lethality. Non-tumorous cells, with the initial pathway intact, can survive. In colon cancer, epigenetic defects in the WRN gene appear to be synthetically lethal with inactivation of TOP1. In particular, irinotecan inactivation of TOP1 was synthetically lethal with deficient expression of the DNA repair WRN gene in patients with colon cancer."
DOCUMENT4 = "A hemangioma or haemangioma is a usually benign vascular tumor derived from blood vessel cell types. The most common form, seen in infants, is an infantile hemangioma, known colloquially as a \"strawberry mark\", most commonly presenting on the skin at birth or in the first weeks of life. A hemangioma can occur anywhere on the body, but most commonly appears on the face, scalp, chest or back. They tend to grow for up to a year before gradually shrinking as the child gets older. A hemangioma may need to be treated if it interferes with vision or breathing or is likely to cause long-term disfigurement. In rare cases internal hemangiomas can cause or contribute to other medical problems. They usually disappear by 10 years of age.[1] The first line treatment option is beta blockers, which are highly effective in the majority of cases. Hemangiomas present at birth are called congenital hemangiomas, while those that form later in life are called infantile hemangiomas. Hemangiomas are benign (noncancerous) vascular tumors, and many different types occur. The correct terminology for these hemangioma types is constantly being updated by the International Society for the Study of Vascular Anomalies (ISSVA).[3] The most common are infantile hemangiomas, and congenital hemangiomas. Infantile hemangiomas are the most common benign tumor found in children. They are made up of blood vessels, often called strawberry marks, and are more common in girls than in boys. Babies that are born early are more likely to have a hemangioma.[4] They usually appear on the skin of infants in the days or weeks after birth. Congenital hemangiomas are present on the skin at birth, unlike infantile hemangiomas, which appear later. They are fully formed at birth, meaning that they do not grow after a child is born, as infantile hemangiomas do. They are less common than infantile hemangiomas. Congenital hemangiomas can be coloured from pink to blue. Diagnosis is usually clinical. Paediatric dermatologists are experts in diagnosing and treating hemangiomas. Depending on the location of the hemangioma, tests such as MRIs or ultrasounds can be done to see how far the hemangioma goes under the skin and whether it affects any internal organs. Hemangiomas usually fade gradually over time, and many do not require treatment. However, hemangiomas that may be disfiguring or that are located at sites that can cause impairment (eyelids, airway) require early treatment intervention, typically with beta blockers. Management options may include Oral beta blockers such as propranolol or atenolol have been used since 2008 and are the first-line treatment of hemangiomas. Beta blockers have repeatedly been shown to be effective and safe in treating hemangiomas that cause complications.[14] Beta blockers work via multiple mechanisms including narrowing the hemangioma's blood vessels, stopping them from proliferating and bringing forward their natural cell death. These correspond with hemangiomas fading and shrinking.[15] Approximately 97\% of hemangiomas respond to propanolol, with patients under 2 months old showing the greatest improvement. Topical beta blockers such as timolol. They are most helpful for thin superficial hemangiomas.[17] These should not be used in conjunction with oral beta blockers given systemic absorption of topical timolol is known to occur"
DOCUMENT5 = "The congenital melanocytic nevus is a type of melanocytic nevus (or mole) found in infants at birth. This type of birthmark occurs in an estimated 1\% of infants worldwide; it is located in the area of the head and neck 15\% of the time. As compared with a melanocytic nevus, congenital melanocytic nevi are usually larger in diameter and may have excess terminal hair, a condition called hypertrichosis. If over 40 cm (16 in) projected adult diameter with hypertrichosis, it is sometimes called giant hairy nevus; more usually these largest forms are known as large or giant congenital melanocytic nevus. The estimated prevalence for the largest forms is 0.002\% of births. Neurocutaneous melanosis is associated with the presence of either giant congenital melanocytic nevi or non-giant nevi of the skin. It is estimated that neurocutaneous melanosis is present in 2\% to 45\% of patients with giant congenital melanocytic nevi. Neurocutaneous melanosis is characterized by the presence of congenital melanocytic nevi on the skin and melanocytic tumors in the leptomeninges of the central nervous system. Large congenital nevi are caused by a mutation in the body's cells that occurs early in embryonic development, usually within the first twelve weeks of pregnancy.[3] Mutations are sometimes found in genes that code for NRAS and KRAS proteins.[4] There is no known method of prevention. Benign congenital nevi can have histological characteristics resembling melanomas, often breaking most if not all of the ABCDE rules. Dermatoscopic findings of the smaller forms of benign congenital nevi can aid in their differentiation from other pigmented neoplasms. Microscopically, congenital melanocytic nevi appear similar to acquired nevi with two notable exceptions. For the congenital nevus, the neval cells are found deeper into the dermis. Also, the deeper nevus cells can be found along with neurovascular bundles, with both surrounding hair follicles, sebaceous glands, and subcutaneous fat. Such annexes and the Subcutaneous tissue can also be hypoplasic or, conversely, present aspects of hamartoma. Surgical excision is the standard of care. Some individuals advocate the use of hair removal laser for the treatment of congenital nevi. While this is likely safe and effective for small congenital nevus, laser removal for larger lesions might pose a liability for the laser surgeon if malignancy developed from a deep (dermal) component of the nevus that is not reached by the laser. Repigmentation after laser treatment of congenital nevi or superficial curettage supports this concern. Many are surgically removed for aesthetics and relief of psychosocial burden, but larger ones are also excised for prevention of cancer, although the benefit is impossible to assess for any individual patient. Proliferative nodules are usually biopsied and are regularly but not systematically found to be benign.[8] Estimates of transformation into melanoma vary from 2-42\% in the literature, but are most commonly considered to be at the low end of that spectrum due to early observer bias. Porokeratotic eccrine ostial and dermal duct nevus (PEODDN) is a skin lesion that resembles a comedonal nevus, but it occurs on the palms and soles where pilosebaceous follicles are normally absent.[1] It is probably transmitted by paradominant transmission. The foundation of diagnosis is histopathology; cornoid lamella with acrosyringia involved is pathognomonic for PEODDN. It is typically linked to eccrine duct dilatation. Differential diagnoses include inflammatory linear verrucous epidermal nevus, porokeratosis plantaris discreta, nevus comedonicus, linear psoriasis, linear epidermal nevus, spiny keratoderma, congenital unilateral punctate porokeratosis, linear porokeratosis, and porokeratosis of Mibelli. There are few choices for treatment. With time, some lesions may spontaneously flatten. Surgery may be a good option for small, isolated lesions. Laser therapy is a great technique because there is very little risk of pigmentary alterations and scarring, especially when using an ultra-pulse CO2 laser. Patients with PEODDN have demonstrated considerable cosmetic improvement with combined erbium/CO2 laser therapy.[10] Topical steroids, retinoids, phototherapy, electrocautery, keratolytics, and cryotherapy are examples of modalities that have not demonstrated any encouraging outcomes. Nevus of Ota is a hyperpigmentation[3] that occurs on the face, most often appearing on the white of the eye. It also occurs on the forehead, nose, cheek, periorbital region, and temple. A Q-switched 1064 nm laser has been successfully used to treat the condition.[7][8] The Q-switched lasers (694 nm ruby, 755 nm Alexandrite or 1064 nm Nd-YAG) with their high peak power and pulse width in nano second range are best suited to treat various epidermal, junctional, mixed and dermal lesions. The Q-switched 1064 nm Nd-YAG is an ideal choice to treat dermal pigment as in nevus of Ota and in darker skin types, as it reduces the risk of epidermal injury and pigmentary alterations. The pigment clearance can be expected to be near total, using multiple treatment sessions, each separated by a minimum of six weeks. The number of treatments required depends on the severity of the lesion. A darker lesion needs more treatments. The outcome also depends to some extent on the power output and quality of the laser system. Last but not least, the skill of the laser surgeon plays a role in achieving early and good clearance. A specific form of conjunctivoplasty may help somewhat. A Gundersen flap, also known as Gundersen's flap, Gundersen's conjunctival flap, or conjunctivoplasty, and often misspelled Gunderson, is a surgical procedure for correcting corneal disease. It involves excising a damaged section of cornea, and replacing it with a section (or \"flap\") of the patient's own conjunctiva."
DOCUMENT6 = "Paget's disease of the breast (also known as mammary Paget's disease) is a rare skin change at the nipple nearly always associated with underlying breast cancer.[2] Paget's disease of the breast was first described by Sir James Paget in 1874.[3] The condition is an uncommon disease accounting for 1 to 4\% of all breast cancers cases.[2] 92\% to 100\% of patients with Paget's disease of the breast have an underlying breast cancer. The condition in itself often appears innocuous, limited to a surface appearance and it is sometimes dismissed, although it is actually indicative of underlying breast cancer. Paget's disease of the breast can affect the nipple and areola: the nipple is typically affected first and then the skin changes spread to the areola. It is common for symptoms to wax and wane. Symptoms typically only affect one breast and may include: Skin: The first symptom is usually an eczema-like rash. The skin of the nipple and areola may be red, itchy, or tingly.[2] After a period of time, the skin may become flaky, scaly, or thickened. Many patients do not visit the doctor because they assume Paget's disease of the breast to be minor contact dermatitis or eczema. Nipple discharge: A discharge, which may be yellow or bloody, may ooze from the area. Nipple changes: The nipple may become inverted. Breast changes: Palpable lumps or masses may be present.[4][6] There may be redness, oozing and crusting, and a sore that does not heal. A person with Paget's disease of the breast may experience signs and symptoms for several months before a diagnosis is made. During a physical examination, the provider will likely conduct a breast examination: evaluating the appearance of the skin on and around the nipples, and feeling for any lumps or areas of thickening in the breast and armpit. Paget's disease of the breast is difficult to diagnose by physical exam alone due to its resemblance to dermatitis and eczema. One helpful differentiator is that eczema tends to affect the areola first, and then the nipple, whereas Paget's disease of the breast typically begins at the nipple and spreads outwards. In addition, nipple eczema is typically responsive to topical steroid application, while Paget's disease of the breast will not improve with topical steroid use. Paget's disease of the breast is a symptom of underlying breast cancer. Treatment is variable and is determined by the type of breast cancer in addition to its staging and prognostic considerations. Management often involves a lumpectomy or mastectomy to surgically remove the tumour.[12] Chemotherapy and/or radiotherapy may also be necessary. Patients with Paget's disease of the breast that has not spread beyond the nipple are often treated with breast-conserving surgery: removal of the cancerous area of the nipple and areola, but conservation of the rest of the breast. Patients then usually undergo radiation therapy after surgery as an adjuvant treatment to prevent recurrence. In most cases, adjuvant treatment is part of the treatment schema. Adjuvant therapy is given to patients with cancer as a secondary form of treatment to minimize the risk of recurrence by targeting undetectable metastases. Whether adjuvant therapy is needed depends upon the type of cancer and its staging. In Paget's disease of the breast, the most common type of adjuvant therapy is radiation following breast-conservative surgery as discussed above. Paget's disease of the breast with underlying breast cancer is primarily treated with mastectomy. In cases of invasive cancer, radical mastectomy is performed: removal of the breast, the lining over the chest muscles, and affected lymph nodes from under the arm. In cases of noninvasive cancers, simple mastectomy are performed in which only the breast with the lining over the chest muscles is removed. Three factors are evaluated when determining prognosis for breast cancer: whether there is a palpable mass, whether lymph nodes have cancer cells in them, and whether there is an underlying metastatic cancer. Prognosis of Paget's disease of the breast with underlying breast cancer depends on these three factors of the underlying cancer. Whether or not a patient has Paget's disease of the breast does not affect their prognosis in the presence of underlying breast cancer. Patients with Paget's disease of the breast and no underlying breast cancer have a 5-year relative survival rate of 82.6\%"
DOCUMENT7 = "In medicine, desmoplasia is the growth of fibrous connective tissue.[1] It is also called a desmoplastic reaction to emphasize that it is secondary to an insult. Desmoplasia may occur around a neoplasm, causing dense fibrosis around the tumor,[1] or scar tissue (adhesions) within the abdomen after abdominal surgery. Desmoplasia is usually only associated with malignant neoplasms, which can evoke a fibrotic response invading healthy tissue. Invasive ductal carcinomas of the breast often have a stellate appearance caused by desmoplastic formations. Cancer begins as cells that grow uncontrollably, usually as a result of an internal change or oncogenic mutations within the cell.[8] Cancer develops and progresses as the microenvironment undergoes dynamic changes.[9] The stromal reaction in cancer is similar to the stromal reaction induced by injury or wound repair: increased extracellular matrix (ECM) and growth factor production and secretion, which consequently cause growth of the tissue.[10] In other words, the body reacts similarly to a cancer as it does to a wound, causing scar-like tissue to be built around the cancer. As such, the surrounding stroma plays a very important role in the progression of cancer. The interaction between cancer cells and surrounding tumor stroma is thus bidirectional, and the mutual cellular support allows for the progression of the malignancy. Stroma contains extracellular matrix components such as proteoglycans and glycosaminoglycans which are highly negatively charged, largely due to sulfated regions, and bind growth factors and cytokines, acting as a reservoir of these cytokines.[5] In tumors, cancer cells secrete matrix degrading enzymes, such as matrix metalloproteinases (MMPs) that, once cleaved and activated, degrade the matrix, thereby releasing growth factors that signal for the growth of cancer cells.[11] MMPs also degrade ECM to provide space for vasculature to grow to the tumor, for the tumor cells to migrate, and for the tumor to continue to proliferate. Desmoplasia is thought to have a number of underlying causes. In the reactive stroma hypothesis, tumor cells cause the proliferation of fibroblasts and subsequent secretion of collagen.[3] The newly secreted collagen is similar to that of collagen in scar formation – acting as a scaffold for infiltration of cells to the site of injury.[12] Furthermore, the cancer cells secrete matrix degrading enzymes to destroy normal tissue ECM thereby promoting growth and invasiveness of the tumor.[3] Cancer associated with a reactive stroma is typically diagnostic of poor prognosis. Diagnosis is by visualisation and dermoscopy.[4] A biopsy is sometimes performed, or the whole lesion surgically removed.[3] The outcome is generally good but there is a small chance of cancerous transformation.[3] Differential diagnosis includes dermatofibroma and melanoma. The blue colour is caused by the pigment being deep in the skin. A blue nevus is a type of coloured mole, typically a single well-defined blue-black bump. A patch blue nevus (also known as an \"acquired dermal melanocytosis\", and \"dermal melanocyte hamartoma\") is a cutaneous condition characterized by a diffusely gray-blue area that may have superimposed darker macules. A blue nevus of Jadassohn–Tièche (also known as a \"common blue nevus\", and \"nevus ceruleus\") is a cutaneous condition characterized by a steel-blue papule or nodule. A deep penetrating nevus is a type of benign melanocytic skin tumor characterized, as its name suggests, by penetration into the deep dermis and/or subcutis. Smudged chromatic is a typical finding. In some cases mitotic figures or atypical melanocytic cytology are seen, potentially mimicking a malignant melanoma. Evaluation by an expert skin pathologist is advisable in some cases to help differentiate from invasive melanoma. A dermatofibroma, or benign fibrous histiocytomas, is a benign nodule in the skin, typically on the legs, elbows or chest of an adult.[3] It is usually painless. It usually ranges from 0.2 to 2 cm in size but larger examples have been reported.[3] It typically results from mild trauma such as an insect bite.[3] Risk factors for developing multiple dermatofibromas include lupus, HIV, blood cancer and some medicines that weaken immunity. Dermatofibromas[4] are hard solitary slow-growing papules (rounded bumps) that appear in a variety of colours, usually brownish to tan. They are often elevated or pedunculated. A dermatofibroma is associated with the dimple sign; by applying lateral pressure, there is a central depression of the dermatofibroma. Although typical dermatofibromas cause little or no discomfort, itching and tenderness can occur. Dermatofibromas can be found anywhere on the body, but most often they are found on the legs and arms.[5] They occur more often in women; the male to female ratio is about 1:4.[6] The age group in which they most commonly occur is 20 to 45 years. Some physicians and researchers believe dermatofibromas form as a reaction to previous injuries such as insect bites or thorn pricks.[6] They are composed of disordered collagen laid down by fibroblasts. Dermatofibromas are classed as benign skin lesions, meaning they are completely harmless, though they may be confused with a variety of subcutaneous tumours.[7] Deep penetrating dermatofibromas may be difficult to distinguish, even histologically, from rare malignant fibrohistocytic tumours like dermatofibrosarcoma protuberans. Dermatofibromas typically have a positive buttonhole sign, or central dimpling in the center. Nodules in skin include dermatofibroma[5] and pyogenic granuloma.[6] Nodules may form on tendons and muscles in response to injury,[7] and are frequently found on vocal cords.[8] They may occur in organs such as the lung,[9] or thyroid,[10] or be a sign in other medical conditions such as rheumatoid arthritis. If a dermatofibroma is large or causes discomfort, your healthcare provider may remove it. Removal is a short in-office procedure. They may use: Steroid injections to reduce pain or lesion size. Dermatofibromas are usually harmless and don't require treatment. Treatment is typically considered if a dermatofibroma is causing pain, irritation, or if it's unsightly. Common treatment options include intralesional steroid injections, surgical removal, or flattening with liquid nitrogen. Carbon dioxide and pulsed-dye laser treatments have been used in the treatment of dermatofibromas. Treatment is generally unnecessary unless the lesion is symptomatic, though excision is recommended for suspicious or atypical cases. If your dermatofibroma is painful or cosmetically bothersome, they may try: Liquid nitrogen (freezing) therapy to reduce the lesion's size."
DOCUMENT8 = "Angiofibroma (AGF) is a descriptive term for a wide range of benign skin or mucous membrane (i.e. the outer membrane lining body cavities such as the mouth and nose) lesions in which individuals have. benign papules, i.e. pinhead-sized elevations that lack visible evidence of containing fluid; nodules, i.e. small firm lumps usually > 1 mm in diameter; and/or, tumors, i.e. masses often regarded as ~8 mm or larger. AGF lesions share common macroscopic (i.e. gross) and microscopic appearances. Grossly, AGF lesions consist of multiple papules, one or more skin-colored to erythematous, dome-shaped nodules, or usually just a single tumor. Microscopically, they consist of spindle-shaped and stellate-shaped cells centered around dilated and thin-walled blood vessels in a background of coarse bundles of collagen (i.e. the main fibrous component of connective tissue). Angiofibromas have been divided into different types but commonly a specific type was given multiple and very different names in different studies. These papule, nodule, and/or tumor lesions occur on the: face and are typically termed fibrous papules; penis and are typically termed pearly penile papules; and underneath a fingernail or toenail and are typically termed periungual angiofibromas. Some of these cutaneous AGF lesions occur in individuals with one or more of 3 different genetic diseases: tuberous sclerosis, multiple endocrine neoplasia type 1, and Birt-Hogg-Dube syndrome.[3] The following are examples of these cutaneous angiofibromas and their alternate names. Fibrous papules are also termed facial angiofibromas and were formerly and incorrectly termed adenoma sebaceum (fibrous papules are unrelated to sebaceous glands[4]). They develop in up to 8\% of the general adult population and occur as 1 to 3[5] pink to red,[4] dome-shaped papules in the central areas of the face, nose, and/or lips.[6] About 75\% of individuals with tuberous sclerosis present with fibrous papules in their infancy or early childhood; when associated with this rare disease, the lesions often occur as multiple papules[5] in symmetrical, butterfly-shaped patterns over both cheeks and the nose.[7] Fibrous papules also occur in individuals with multiple endocrine neoplasia type 1 (a study done in Japan found that 43\% of individuals with this genetic disease bore facial angiofibromas)[8] and, uncommonly, in individuals with Birt-Hogg-Dube syndrome. Pearly penile papules are also termed papillae coronae glandis and hirsutoid papillomas. The condition of having such papules or papillae is called hirsuties papillaris coronae glandis or papillomatosis coronae glandis or papillomatosis coronae penis.[10] These lesions develop in up to 30\% of males during their puberty or, less commonly, early adulthood. They typically occur as numerous white-colored to skin-colored papules located circumferentially around the corona of the penis or, less commonly, the ventromedial aspect of the corona near the penis's frenulum.[11] (Vestibular papillomatosis, also named hirsutoid vulvar papillomas, vulvar squamous papillomatosis, micropapillomatosis labialis, and squamous vestibular micropapilloma, is the female equivalent of pearly penile papules in men. Periungual angiofibromas are also termed Koenen's tumors, periungual fibromas, and subungual fibromas.[13] In addition, these tumors were formerly regarded as a type of acral angiofibroma (see below description).[14] These lesions present as multiple nodules or tumors under multiple finger and/or toe nails of individuals with tuberous sclerosis[4] or in one case the Birt-Hogg-Dube syndrome.[15] Periungual angiofibromas have also been reported to occur in individuals that do not have these genetic diseases.[16] Periungual angiofibromas tumors can be highly mutilating finger/toe-nail lesions. Oral fibromas are also termed irritation fibromas, focal fibrous hyperplasia, and traumatic fibromas.[17] These lesions are nodules that occur on the buccal mucosa (i.e. mucous membranes lining the cheeks and back of the lips) or lateral tongue.[18] They may be irritating or asymptomatic and are the most common tumor-like lesions in the oral cavity. Oral fibromas are not neoplasms; they are hyperplastic (i.e. overgrowth) reactions of fibrous tissue to local trauma or chronic irritation. Nasopharyngeal angiofibromas, also termed juvenile nasopharyngeal angiofibromas, fibromatous hamartomas, or angiofibromatous hamartoma of the nasal cavity, are large benign tumors (average size 5.9 cm in one study) that develop almost exclusively in males aged 9 to 36 years old. They commonly arise in the nasopharynx (i.e. upper part of the throat that lies behind the nose) and typically have attachments to the sphenopalatine foramen, clivus, and/or root of the pterygoid processes of the sphenoid bone. These tumors may expand into various other nearby structures including the cranial cavity.[20] Nasopharyngeal angiofibromas are highly vascularized tumors consisting of fibroblasts (i.e. connective tissue cells) in a dense collagen matrix (i.e. tissue background). Studies have suggested that these tumors are due to the expression of male sex hormones (i.e. androgens and progesterones), genetic factors, molecular alterations (i.e. changes in the normal characteristics of cells that lead to abnormal cell growth), and/or human papillomavirus infection. Angiofibroma treatment depends on the type and location of the tumor. For facial angiofibromas, options include surgical removal, laser therapy, and topical medications like rapamycin or topical beta-blockers. For nasopharyngeal angiofibromas, surgical resection is the primary treatment, often combined with embolization to reduce blood loss. In some cases, radiation therapy may be used, particularly for recurrent or unresectable tumors.\nFacial Angiofibromas:\nSurgical Removal: Excision, dermabrasion, shave excision, or electrosurgery can be used to remove the angiofibroma.\nLaser Therapy: Various lasers like ablative CO2 lasers and pulsed dye lasers (PDL) are effective in treating facial angiofibromas, particularly those with a vascular component. Topical Medications: Topical rapamycin (an mTOR inhibitor) and topical beta-blockers (like timolol) have shown promise in reducing the size and appearance of facial angiofibromas. Cryotherapy: Freezing the angiofibroma with liquid nitrogen is another option. Juvenile Nasopharyngeal Angiofibroma (JNA):\nSurgical Resection: The primary treatment is surgical removal, often with endoscopic or minimally invasive approaches, especially for localized tumors.\nEmbolization: Before surgery, interventional radiologists may embolize the blood vessels supplying the tumor to reduce bleeding during the procedure.\nRadiation Therapy: Radiation therapy may be used for recurrent or unresectable tumors, or to reduce tumor size before surgery.\nStereotactic Radiosurgery: A more precise form of radiation therapy that can be used for advanced cases or those with intracranial extension.\nImportant Considerations: Benign Nature: Angiofibromas are benign, meaning they are not cancerous.\nRecurrence: Angiofibromas can have high recurrence rates, so follow-up care and potential further treatment may be necessary.\nExpertise: Treatment for JNA, especially for complex cases with intracranial involvement, should be performed by experienced surgeons. Topical treatment with sirolimus has been found to decrease the size of facial angiofibromas and accelerate the resolution of erythema"

In [11]:
documents = [DOCUMENT1,DOCUMENT2,DOCUMENT3,DOCUMENT4,DOCUMENT5,DOCUMENT6,DOCUMENT7, DOCUMENT8]

In [12]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry

from google.genai import types


# Define a helper to retry when per-minute quota is reached.
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})


class GeminiEmbeddingFunction(EmbeddingFunction):
    # Specify whether to generate embeddings for documents, or queries
    document_mode = True

    @retry.Retry(predicate=is_retriable)
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = "retrieval_document"
        else:
            embedding_task = "retrieval_query"

        response = client.models.embed_content(
            model="models/text-embedding-004",
            contents=input,
            config=types.EmbedContentConfig(
                task_type=embedding_task,
            ),
        )
        return [e.values for e in response.embeddings]

In [13]:
import chromadb

DB_NAME = "isic_skin_lesion_literature"

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

db.add(documents=documents, ids=[str(i) for i in range(len(documents))])

In [14]:
db.count()

8

In [15]:
# Define get_answer function with RAG and JSON report generation
def get_answer(query: str, lesion_type: str = None, anatom_site: str = None) -> str:
    embed_fn.document_mode = False
    search_query = query
    if lesion_type:
        search_query = f"{lesion_type}: {query}"
        if anatom_site and anatom_site != "unknown":
            search_query = f"{lesion_type} at {anatom_site}: {query}"
    
    result = db.query(query_texts=[search_query], n_results=2)  # Get top 2 documents for better context
    passages = result["documents"][0] if result and result["documents"] and result["documents"][0] else []
    
    query_oneline = query.replace("\n", " ")
    prompt = f"""You are a helpful and informative dermatology assistant that answers questions using the reference passages below.
    Be comprehensive, including all relevant background information, and strike a friendly, professional tone.
    If the query asks for a diagnosis report, next steps, or treatment options, you MUST generate a JSON object with the keys: 'lesion_type', 'next_steps', and 'treatment_options'. The 'anatom_site' should also be included if available. The 'confidence' key is optional.
    If the passages are irrelevant, provide a general response based on dermatological knowledge or indicate lack of specific information.
    If no lesion_type is provided and a report is requested, inform the user that the lesion type is needed.

    QUESTION: {query_oneline}
    """

    for passage in passages:
        passage_oneline = passage.replace("\n", " ")
        prompt += f"PASSAGE: {passage_oneline}\n"

    if not passages:
        prompt += "No relevant passages found. Provide a general dermatological response if applicable.\n"

    # If the query is about diagnosis, next steps, or treatment, generate a JSON report
    if any(keyword in query.lower() for keyword in ["report", "next steps", "treatment", "diagnosis"]):
        if lesion_type:
            prompt += f"""
            You MUST now generate a JSON diagnosis report for a skin lesion classified as '{lesion_type}' located at '{anatom_site or 'unknown'}'.
            The JSON object MUST have the following keys: 'lesion_type', 'anatom_site', 'next_steps', and 'treatment_options'.
            The 'confidence' key can be included if you have a confidence level (e.g., "high", "moderate", "low").
            Ensure the values for these keys are informative and professionally worded based on the provided passages.
            The JSON object should be the ONLY output if a report is requested. Do not include any surrounding text.
            Example JSON output:
            ```json
            {{
              "lesion_type": "Basal Cell Carcinoma",
              "anatom_site": "Nose",
              "next_steps": "Biopsy to confirm diagnosis and determine subtype.",
              "treatment_options": ["Surgical excision", "Mohs surgery", "Radiation therapy (for certain cases)"]
            }}
            ```
            """
        else:
            return "Please provide the lesion type to generate a diagnosis report."

    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt
        )
        response_text = response.text
        # print(f"Raw Gemini response: {response_text}")  # Debug response

        if response_text.startswith("```json") and response_text.endswith("```"):
            json_str = response_text[7:-3].strip()
            try:
                json_response = json.loads(json_str)
                return json.dumps(json_response, indent=2)  # Return formatted JSON
            except json.JSONDecodeError as e:
                print(f"JSON parsing error: {str(e)}")
                return response_text  # Fallback to text
        else:
            return response_text
    except Exception as e:
        print(f"Error in Gemini API call: {str(e)}")
        return f"Sorry, I couldn't generate a response. Please try again or consult a dermatologist."

In [16]:
# Function to process user-provided image
def process_image(image_path, anatom_site="unknown"):
    if not os.path.exists(image_path):
        print(f"Image not found at {image_path}. Using a default image.")
        image_path = os.path.join(IMAGES_DIR, "img_1.jpg")
        if not os.path.exists(image_path):
            raise FileNotFoundError("Default image not found.")
    
    try:
        image = tf.io.read_file(image_path)
        image_decoded = tf.image.decode_jpeg(image, channels=3)
        print(f"Decoded image shape: {image_decoded.shape}")  # Debug shape
        image_array = image_decoded.numpy()  # For visualization
        image_resized = tf.image.resize(image_decoded, [224, 224])
        image_normalized = image_resized / 255.0
        image_batched = tf.expand_dims(image_normalized, axis=0)
        prediction = model.predict(image_batched)
        lesion_type_idx = np.argmax(prediction)
        confidence = prediction[0][lesion_type_idx]
        if le is None:
            raise ValueError("LabelEncoder is not initialized. Check metadata loading.")
        lesion_type = le.inverse_transform([lesion_type_idx])[0]
        
        # Visualize the image
        plt.imshow(image_array)
        plt.title(f"Predicted: {lesion_type} ({confidence:.2f})")
        plt.axis('off')
        plt.savefig('/kaggle/working/prediction_image.png')
        plt.close()
        
        return lesion_type, confidence
    except Exception as e:
        print(f"Error processing image: {str(e)}")
        raise

In [17]:
# Define the ChatState model
class ChatState(BaseModel):
    messages: List[Dict[str, str]]
    lesion_type: str
    confidence: float
    anatom_site: str

# Fallback function
def fallback():
    return "Sorry, my knowledge is limited to skin lesions. Please consult a medical practitioner or ask about the diagnosis."

In [18]:
# RAG QA Node
def rag_qa_node(state: ChatState) -> ChatState:
    if not state.messages:
        return state
    
    query = state.messages[-1]["content"].lower()
    lesion_type = state.lesion_type
    anatom_site = state.anatom_site
    confidence = state.confidence
    
    if "lesion type" in query:
        answer = f"The lesion type is {lesion_type}."
    elif "confidence" in query:
        answer = f"The confidence level is {confidence:.2f}."
    elif "next steps" in query or "treatment" in query or "report" in query:
        answer = get_answer(query, lesion_type, anatom_site)
    else:
        answer = get_answer(query, lesion_type, anatom_site) or fallback()
    
    updated_messages = state.messages + [{"role": "assistant", "content": answer}]
    return ChatState(messages=updated_messages, lesion_type=lesion_type, confidence=confidence, anatom_site=anatom_site)

In [19]:
print("""
Instructions for Judges:
1. The dataset includes more than 100 images in /kaggle/input/test-dataset/test_images/ (e.g., img_1.jpg, img_13.jpg....).
2. To test just make call to start_chat() with the image_name and anatom_site(optional) and the bot
    will give you the diagnosis.
3. The notebook demonstrates CNN inference, structured_output, RAG-based analysis, and a LangGraph chatbot.
5. A pre-trained model (skin_lesion_model.h5) ensures quick execution.
    """)
print("Disclaimer: This AI-generated advice is for demonstration and should be verified by a medical professional.")


Instructions for Judges:
1. The dataset includes more than 100 images in /kaggle/input/test-dataset/test_images/ (e.g., img_1.jpg, img_13.jpg....).
2. To test just make call to start_chat() with the image_name and anatom_site(optional) and the bot
    will give you the diagnosis.
3. The notebook demonstrates CNN inference, structured_output, RAG-based analysis, and a LangGraph chatbot.
5. A pre-trained model (skin_lesion_model.h5) ensures quick execution.
    
Disclaimer: This AI-generated advice is for demonstration and should be verified by a medical professional.


In [20]:
# Initialize LangGraph
graph = StateGraph(ChatState)
graph.add_node("RAG_QA", rag_qa_node)
graph.set_entry_point("RAG_QA")
graph.set_finish_point("RAG_QA")
app = graph.compile()

# Function to start the chat
def start_chat(user_image_filename="img_3", user_anatom_site="unknown"):
    
    user_image_path = os.path.join(IMAGES_DIR, f"{user_image_filename}.jpg")
    
    try:
        lesion_type, confidence = process_image(user_image_path, user_anatom_site)
        print(f"Predicted Lesion Type: {lesion_type}, Confidence: {confidence:.2f}")
    except Exception as e:
        print(f"Failed to process image: {str(e)}")
        return
    
    state = ChatState(messages=[], lesion_type=lesion_type, confidence=confidence, anatom_site=user_anatom_site)
    
    queries = [
        "Generate a diagnosis report"
    ]
    
    for query in queries:
        print(f"You: {query}")
        state_dict = {
            "messages": state.messages + [{"role": "user", "content": query}],
            "lesion_type": lesion_type,
            "confidence": confidence,
            "anatom_site": user_anatom_site
        }
        result = app.invoke(state_dict)
        bot_output = result['messages'][-1]['content']
        # print(json.dumps(bot_output, indent=2, ensure_ascii=False))
        # display(JSON(bot_output))
        print(f"Bot: {result['messages'][-1]['content']}")
        state = ChatState(messages=result['messages'], lesion_type=lesion_type, confidence=confidence, anatom_site=user_anatom_site)
    

if __name__ == "__main__":
    print("⛑️ Hello, I’m DermaDetective—your trusted AI partner in skin lesion detection and analysis. How can I assist you today?")
    start_chat("img_1", "lower back")
    print("-----                  -------")
    start_chat("img_54", "neck")
    print("-----                  -------")
    start_chat("img_120", "lower back")

⛑️ Hello, I’m DermaDetective—your trusted AI partner in skin lesion detection and analysis. How can I assist you today?
Decoded image shape: (1024, 1024, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted Lesion Type: basal cell carcinoma, Confidence: 0.33
You: Generate a diagnosis report
Bot: {
  "lesion_type": "Basal Cell Carcinoma",
  "anatom_site": "Lower Back",
  "next_steps": "Biopsy to confirm diagnosis and determine subtype. Further evaluation to rule out metastasis, though rare.",
  "treatment_options": [
    "Surgical excision",
    "Mohs surgery",
    "Electrodesiccation and curettage",
    "Cryosurgery",
    "Topical chemotherapy (5-fluorouracil or Imiquimod)",
    "Photodynamic therapy",
    "Radiation therapy",
    "Vismodegib or Sonidegib (for advanced cases)",
    "Itraconazole"
  ]
}
-----                  -------
Decoded image shape: (1024, 1024, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
Predicted Lesion Type: basal cell carcinoma, Confidence: 0.34
You: Generate a diagn